In [ ]:
# In[1]:

import pandas as pd
import pytz

# Reuse variables across steps as required
tz = pytz.timezone('Asia/Shanghai')

# Load CSV files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Helper to produce min/max timestamps (unix + localized string)
def ts_min_max_info(df, ts_col='timestamp'):
    total = int(df.shape[0])
    # ensure timestamps are numeric
    s = pd.to_numeric(df[ts_col], errors='coerce')
    mn = int(s.min()) if not s.isna().all() else None
    mx = int(s.max()) if not s.isna().all() else None
    if mn is not None:
        mn_local = pd.to_datetime(mn, unit='s', utc=True).tz_convert(tz)
        mx_local = pd.to_datetime(mx, unit='s', utc=True).tz_convert(tz)
        mn_str = mn_local.strftime('%Y-%m-%d %H:%M:%S %Z')
        mx_str = mx_local.strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        mn_str = mx_str = None
    return total, mn, mn_str, mx, mx_str

# Summaries and top lists for metric.csv
metric_total, metric_min_ts, metric_min_local, metric_max_ts, metric_max_local = ts_min_max_info(metric_df, 'timestamp')
metric_distinct_cmdb = int(metric_df['cmdb_id'].nunique())
metric_top_cmdb = metric_df.groupby('cmdb_id', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
metric_distinct_kpi = int(metric_df['kpi_name'].nunique())
metric_top_kpi = metric_df.groupby('kpi_name', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

metric_summary = pd.DataFrame([{
    'total_rows': metric_total,
    'min_timestamp_unix': metric_min_ts,
    'min_timestamp_local': metric_min_local,
    'max_timestamp_unix': metric_max_ts,
    'max_timestamp_local': metric_max_local,
    'distinct_cmdb_id': metric_distinct_cmdb,
    'distinct_kpi_name': metric_distinct_kpi
}])

# Summaries and top lists for trace.csv
trace_total, trace_min_ts, trace_min_local, trace_max_ts, trace_max_local = ts_min_max_info(trace_df, 'timestamp')
trace_distinct_cmdb = int(trace_df['cmdb_id'].nunique())
trace_top_cmdb = trace_df.groupby('cmdb_id', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
trace_distinct_trace = int(trace_df['trace_name'].nunique())
trace_top_trace = trace_df.groupby('trace_name', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

trace_summary = pd.DataFrame([{
    'total_rows': trace_total,
    'min_timestamp_unix': trace_min_ts,
    'min_timestamp_local': trace_min_local,
    'max_timestamp_unix': trace_max_ts,
    'max_timestamp_local': trace_max_local,
    'distinct_cmdb_id': trace_distinct_cmdb,
    'distinct_trace_name': trace_distinct_trace
}])

# Summaries and top lists for log.csv
log_total, log_min_ts, log_min_local, log_max_ts, log_max_local = ts_min_max_info(log_df, 'timestamp')
log_distinct_cmdb = int(log_df['cmdb_id'].nunique())
log_top_cmdb = log_df.groupby('cmdb_id', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
log_distinct_logname = int(log_df['log_name'].nunique())
log_top_logname = log_df.groupby('log_name', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)

log_summary = pd.DataFrame([{
    'total_rows': log_total,
    'min_timestamp_unix': log_min_ts,
    'min_timestamp_local': log_min_local,
    'max_timestamp_unix': log_max_ts,
    'max_timestamp_local': log_max_local,
    'distinct_cmdb_id': log_distinct_cmdb,
    'distinct_log_name': log_distinct_logname
}])

# Summaries and top lists for error_logs.csv
error_total, error_min_ts, error_min_local, error_max_ts, error_max_local = ts_min_max_info(error_df, 'timestamp')
error_distinct_cmdb = int(error_df['cmdb_id'].nunique())
error_top_cmdb = error_df.groupby('cmdb_id', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(20).reset_index(drop=True)
# Top 10 most frequent message strings
error_top_msgs = error_df.groupby('message', sort=False).size().reset_index(name='count').sort_values('count', ascending=False).head(10).reset_index(drop=True)

error_summary = pd.DataFrame([{
    'total_rows': error_total,
    'min_timestamp_unix': error_min_ts,
    'min_timestamp_local': error_min_local,
    'max_timestamp_unix': error_max_ts,
    'max_timestamp_local': error_max_local,
    'distinct_cmdb_id': error_distinct_cmdb
}])

# Display the compact summaries and top lists (limited as requested)
metric_summary, metric_top_cmdb.head(20), metric_top_kpi.head(20), \
trace_summary, trace_top_cmdb.head(20), trace_top_trace.head(20), \
log_summary, log_top_cmdb.head(20), log_top_logname.head(20), \
error_summary, error_top_cmdb.head(20), error_top_msgs.head(10)

```
Out[1]:
```
summary = (
    "Summary of executed CSV inspections (timestamps shown in Asia/Shanghai, UTC+8):\n\n"
    "metric.csv:\n"
    "- Rows: 319,694\n"
    "- Time range: 2022-03-20 17:00:00 CST to 2022-03-20 18:00:00 CST\n"
    "- Distinct cmdb_id: 59\n"
    "- Top cmdb_id by row count (top examples): adservice (20,801), adservice2 (20,313), frontend2-0 (11,956), "
    "frontend-2/ frontend-1/ frontend-0 (11,468 each). (Top 20 computed.)\n"
    "- Distinct kpi_name: 1,368\n"
    "- Top kpi_name examples: many container.node-6.* metrics (several ~2,196 occurrences) and container.node-6 memory/fs metrics (~2,074 occurrences).\n\n"
    "trace.csv:\n"
    "- Rows: 34,376\n"
    "- Time range: 2022-03-20 17:30:00 CST to 2022-03-20 17:59:00 CST\n"
    "- Distinct cmdb_id: 40\n"
    "- Top cmdb_id by row count: frontend-1 (2,736), frontend-2 (2,632), frontend-0 (2,524), checkoutservice-2 (2,352), checkoutservice-0 (2,340), checkoutservice-1 (2,296). (Top 20 computed.)\n"
    "- Distinct trace_name: 200\n"
    "- Top trace_name examples: traces from frontend pods (e.g., trace.from_frontend-1.* at 624 each), trace.self.* series (480 each).\n\n"
    "log.csv:\n"
    "- Rows: 1,828\n"
    "- Time range: 2022-03-20 17:30:00 CST to 2022-03-20 18:00:00 CST\n"
    "- Distinct cmdb_id: 31\n"
    "- Top cmdb_id examples (each around 60–62 rows): cartservice-2, cartservice-1, cartservice-0, currencyservice-*, frontend-*, etc. (Top 20 computed.)\n"
    "- Distinct log_name: 2 (log.error_count and log.row_count), each with 914 occurrences.\n\n"
    "error_logs.csv:\n"
    "- Rows: 0 (empty)\n"
    "- No timestamps, cmdb_id entries, or messages present.\n\n"
    "Overall notes: metric data is the largest set (319k rows) covering 1 hour; trace and log data cover a 30–30+ minute window. Error log file contains no records. Top identifiers are dominated by adservice/adservice2 and frontend/checkoutservice pods; many prominent KPI names are container.node-6 related."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   total_rows  min_timestamp_unix      min_timestamp_local  max_timestamp_unix      max_timestamp_local  distinct_cmdb_id  distinct_kpi_name
0      319694          1647766800  2022-03-20 17:00:00 CST          1647770400  2022-03-20 18:00:00 CST                59               1368,                      cmdb_id  count
0                  adservice  20801
1                 adservice2  20313
2                frontend2-0  11956
3                 frontend-2  11468
4                 frontend-1  11468
5                 frontend-0  11468
6          checkoutservice-1   8540
7         checkoutservice2-0   8540
8          checkoutservice-2   8296
9          checkoutservice-0   8296
10   productcatalogservice-1   7320
11   productcatalogservice-0   7320
12   productcatalogservice-2   7076
13  productcatalogservice2-0   6344
14  recommendationservice2-0   6222
15             cartservice-2   6100
16             cartservice-0   6100
17            cartservice2-0   6100
18   recommendationservice-1   5978
19   recommendationservice-2   5978,                                              kpi_name  count
0   container.node-6.container_network_receive_pac...   2196
1   container.node-6.container_network_transmit_pa...   2196
2   container.node-6.container_network_receive_MB....   2196
3   container.node-6.container_network_receive_pac...   2196
4   container.node-6.container_network_receive_err...   2196
5   container.node-6.container_network_transmit_er...   2196
6   container.node-6.container_network_transmit_pa...   2196
7   container.node-6.container_network_transmit_MB...   2196
8              container.node-6.container_memory_swap   2074
9       container.node-6.container_fs_reads./dev/vda1   2074
10   container.node-6.container_memory_working_set_MB   2074
11     container.node-6.container_fs_inodes./dev/vda1   2074
12        container.node-6.container_file_descriptors   2074
13     container.node-6.container_memory_max_usage_MB   2074
14              container.node-6.container_memory_rss   2074
15         container.node-6.container_memory_usage_MB   2074
16    container.node-6.container_tasks_state.sleeping   2074
17     container.node-6.container_tasks_state.running   2074
18   container.node-6.container_tasks_state.iowaiting   2074
19  container.node-6.container_fs_sector_reads./de...   2074,    total_rows  min_timestamp_unix      min_timestamp_local  max_timestamp_unix      max_timestamp_local  distinct_cmdb_id  distinct_trace_name
0       34376          1647768600  2022-03-20 17:30:00 CST          1647770340  2022-03-20 17:59:00 CST                40                  200,                     cmdb_id  count
0                frontend-1   2736
1                frontend-2   2632
2                frontend-0   2524
3         checkoutservice-2   2352
4         checkoutservice-0   2340
5         checkoutservice-1   2296
6   productcatalogservice-2   1176
7   productcatalogservice-0   1168
8   productcatalogservice-1   1164
9               frontend2-0   1080
10       checkoutservice2-0   1080
11  recommendationservice-0    840
12  recommendationservice-1    840
13  recommendationservice-2    840
14            cartservice-0    816
15            cartservice-2    816
16            cartservice-1    808
17        currencyservice-1    716
18        currencyservice-2    712
19        currencyservice-0    708,                                     trace_name  count
0              trace.from_frontend-1.row_count    624
1             trace.from_frontend-1.error_rate    624
2           trace.from_frontend-1.duration_p95    624
3          trace.from_frontend-1.duration_mean    624
4              trace.from_frontend-2.row_count    598
5             trace.from_frontend-2.error_rate    598
6           trace.from_frontend-2.duration_p95    598
7          trace.from_frontend-2.duration_mean    598
8          trace.from_frontend-0.duration_mean    571
9           trace.from_frontend-0.duration_p95    571
10            trace.from_frontend-0.error_rate    571
11             trace.from_frontend-0.row_count    571
12                    trace.self.duration_mean    480
13                     trace.self.duration_p95    480
14                       trace.self.error_rate    480
15                        trace.self.row_count    480
16  trace.from_checkoutservice-2.duration_mean    471
17   trace.from_checkoutservice-2.duration_p95    471
18     trace.from_checkoutservice-2.error_rate    471
19      trace.from_checkoutservice-2.row_count    471,    total_rows  min_timestamp_unix      min_timestamp_local  max_timestamp_unix      max_timestamp_local  distinct_cmdb_id  distinct_log_name
0        1828          1647768600  2022-03-20 17:30:00 CST          1647770400  2022-03-20 18:00:00 CST                31                  2,                     cmdb_id  count
0             cartservice-2     62
1             cartservice-1     62
2             cartservice-0     62
3         currencyservice-0     62
4         currencyservice-2     62
5         currencyservice-1     62
6                frontend-0     62
7   recommendationservice-1     62
8   productcatalogservice-1     62
9                frontend-1     62
10  recommendationservice-2     62
11        checkoutservice-0     60
12              adservice-0     60
13              adservice-1     60
14              adservice-2     60
15        shippingservice-1     60
16           emailservice-2     60
17           emailservice-1     60
18        checkoutservice-1     60
19        checkoutservice-2     60,           log_name  count
0  log.error_count    914
1    log.row_count    914,    total_rows min_timestamp_unix min_timestamp_local max_timestamp_unix max_timestamp_local  distinct_cmdb_id
0           0               None                None               None                None                 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [message, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse metric_df if exists, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# 1) Build allowed cmdb_id list (nodes, pods, services) from problem statement
nodes = [f"node-{i}" for i in range(1, 7)]

pods = [
    "frontend-0","frontend-1","frontend-2","frontend2-0",
    "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
    "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
    "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
    "adservice-0","adservice-1","adservice-2","adservice2-0",
    "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
    "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
    "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
    "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
    "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
]

services = [
    "frontend","shippingservice","checkoutservice","currencyservice",
    "adservice","emailservice","cartservice","productcatalogservice",
    "recommendationservice","paymentservice"
]

allowed_cmdb = set(nodes + pods + services)

# 1) Restrict metric_df to allowed cmdb_id values
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()

# Ensure numeric value column
metric_allowed['value'] = pd.to_numeric(metric_allowed['value'], errors='coerce')

# 2) Compute global P95 and P5 for each (cmdb_id, kpi_name) using ALL available rows (no time filtering)
grp = metric_allowed.groupby(['cmdb_id', 'kpi_name'])['value']
thresholds_p95 = grp.quantile(0.95).rename('P95')
thresholds_p05 = grp.quantile(0.05).rename('P5')
thresholds = pd.concat([thresholds_p95, thresholds_p05], axis=1).reset_index()

# 3) Filter metric_allowed to incident window
start_ts = 1647768600
end_ts = 1647770400
metric_window = metric_allowed[(metric_allowed['timestamp'] >= start_ts) & (metric_allowed['timestamp'] <= end_ts)].copy()

# If metric_window is empty, produce empty result
if metric_window.shape[0] == 0:
    top20_pairs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','row_count_in_window','anomaly_count','earliest_anomaly_ts',
        'max_value_in_window','min_value_in_window','P95','P5'
    ])
else:
    # 4) Merge thresholds into window rows
    merged = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')

    # Define anomaly: value > P95 OR value < P5; if thresholds are NaN then anomaly=False
    merged['anomaly'] = False
    has_thresh = merged['P95'].notna() & merged['P5'].notna()
    merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh, 'value'] > merged.loc[has_thresh, 'P95']) | (merged.loc[has_thresh, 'value'] < merged.loc[has_thresh, 'P5'])

    # Aggregate required metrics per (cmdb_id, kpi_name)
    group_keys = ['cmdb_id','kpi_name']
    row_count = merged.groupby(group_keys).size().rename('row_count_in_window')
    anomaly_count = merged.groupby(group_keys)['anomaly'].sum().astype(int).rename('anomaly_count')
    max_in_window = merged.groupby(group_keys)['value'].max().rename('max_value_in_window')
    min_in_window = merged.groupby(group_keys)['value'].min().rename('min_value_in_window')
    # earliest anomaly timestamp (unix seconds) per group; if no anomaly, will be NaN
    anomalies = merged[merged['anomaly']]
    if anomalies.shape[0] > 0:
        earliest_anom_ts = anomalies.groupby(group_keys)['timestamp'].min().rename('earliest_anomaly_ts_unix')
    else:
        earliest_anom_ts = pd.Series(dtype='float64', name='earliest_anomaly_ts_unix')

    # thresholds per group (same for all rows in group) - take first non-null
    p95_series = thresholds.set_index(['cmdb_id','kpi_name'])['P95']
    p5_series = thresholds.set_index(['cmdb_id','kpi_name'])['P5']

    # Combine aggregated series into a DataFrame
    result = pd.concat([row_count, anomaly_count, max_in_window, min_in_window], axis=1).reset_index()

    # Join earliest anomaly ts
    earliest_anom_ts = earliest_anom_ts.reset_index()
    result = result.merge(earliest_anom_ts, on=group_keys, how='left')

    # Join thresholds
    result = result.merge(thresholds, on=group_keys, how='left')

    # Convert earliest_anomaly_ts_unix to localized string in Asia/Shanghai; keep None for NaN
    def to_local_str(x):
        if pd.isna(x):
            return None
        try:
            return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        except Exception:
            return None

    result['earliest_anomaly_ts'] = result['earliest_anomaly_ts_unix'].apply(to_local_str)

    # Select and order columns as requested
    result = result[['cmdb_id','kpi_name','row_count_in_window','anomaly_count','earliest_anomaly_ts',
                     'max_value_in_window','min_value_in_window','P95','P5']]

    # 5) Top 20 pairs by anomaly_count desc
    top20_pairs = result.sort_values(['anomaly_count','row_count_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display compact result (top 20)
top20_pairs

```
Out[2]:
```
Summary of the anomaly scan (global P95/P5 thresholds computed across all available metric data; incident window = 1647768600 … 1647770400, i.e., 2022-03-20 17:30:00–18:00:00 CST):

- Scope: analysis limited to allowed components (nodes node-1..node-6, listed pods and services). Thresholds (P95/P5) were computed globally per (cmdb_id, kpi_name) using all metric rows, then applied to the incident window.

- Overall finding: the top (cmdb_id, kpi_name) pairs show modest numbers of anomaly points within the window. The maximum anomaly_count observed in the top 20 is 6, with several pairs showing 5 anomalies.

- Top anomalous pairs (high-level):
  1. checkoutservice-2 — container.node-5.container_memory_working_set_MB: anomaly_count=6, earliest anomaly 2022-03-20 17:41:00 CST, in-window rows=31, values ranged ~25.46–26.93 MB, P95=26.90625, P5=25.765625.
  2. emailservice-1 — (mesh destination metrics toward checkoutservice.emailservice.*): multiple mesh metrics each with anomaly_count=6, earliest anomaly 2022-03-20 17:37:00 CST, large numeric ranges (examples: counts ~865–5625 with P95s in the thousands; other mesh rates with small fractional values).
  3. frontend-1 — container.node-5.container_memory_usage_MB: anomaly_count=6, earliest anomaly 2022-03-20 17:32:00 CST, in-window values ~28.44–29.96 MB, P95≈29.955, P5≈28.965.
  4. frontend2-0 — container.node-6.fs_usage_MB (/dev/vda1): anomaly_count=6, earliest anomaly 17:51, wide min→max (~25.25–72.35 MB) with P95≈71.54 and P5≈26.22 (indicates a large jump late in the window).
  5. node-2/system.mem.free and node-3/system.load.15/system.mem.free: node-level memory and load metrics also appear among top anomalies (anomaly_count=6), earliest anomalies around 17:31–17:43.

- Other notable pairs in top 20 include adservice JVM/memory-related metrics, network transmit metrics on adservice2-0, and various container CPU or cfs/cpu_usage series — mostly anomaly_count 5.

- Interpretation: anomalies are spread across several components: container memory metrics (node-5 and node-6 related), mesh request/response metrics involving emailservice→checkoutservice paths, frontend pods, and some node-level memory/load metrics. The highest anomaly count is small (6 out of ~31 samples), but some metrics show large jumps (e.g., fs usage and mesh counts) that warrant inspection.

- Recommended next steps:
  - Inspect the listed top components (checkoutservice-2, emailservice-1, frontend-1/2/2-0, adservice/adservice2-0, node-2/node-3) and their recent logs/traces around the earliest anomaly timestamps.
  - For metrics with large value jumps (e.g., filesystem usage or huge mesh counts), check pod logs, recent deployments, and node disk/memory status for correlated events.
  - Correlate with trace/log data and error_logs (error_logs is empty) to find probable root cause.

If you want, I can produce a compact tabular excerpt (the same top-20 table) formatted for quick review or expand on any specific (cmdb_id, kpi_name) row.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  row_count_in_window  anomaly_count      earliest_anomaly_ts  max_value_in_window  min_value_in_window            P95             P5
0   checkoutservice-2   container.node-5.container_memory_working_set_MB                   31              6  2022-03-20 17:41:00 CST            26.929688            25.462891      26.906250      25.765625
1      emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6  2022-03-20 17:37:00 CST          3812.500000           865.000000    2595.000000    1257.500000
2      emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6  2022-03-20 17:37:00 CST             0.004725             0.001050       0.003150       0.001575
3      emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6  2022-03-20 17:37:00 CST          5625.000000          1250.000000    3750.000000    1875.000000
4          frontend-1         container.node-5.container_memory_usage_MB                   31              6  2022-03-20 17:32:00 CST            29.960938            28.435547      29.955078      28.964844
5          frontend-2  mesh.destination.unknown.frontend.istio_reques...                   31              6  2022-03-20 17:31:00 CST             0.087250             0.011000       0.068575       0.022000
6         frontend2-0   container.node-6.container_fs_usage_MB./dev/vda1                   31              6  2022-03-20 17:51:00 CST            72.347656            25.253906      71.539062      26.222656
7              node-2                                    system.mem.free                   31              6  2022-03-20 17:43:00 CST         10175.000000          9734.000000   10110.000000    9754.000000
8              node-3                                     system.load.15                   31              6  2022-03-20 17:38:00 CST             1.050000             0.620000       0.980000       0.660000
9              node-3                                    system.mem.free                   31              6  2022-03-20 17:31:00 CST         10527.000000         10062.000000   10477.000000   10072.000000
10          adservice  runtime.java_lang_GarbageCollector_LastGcInfo_...                   31              5  2022-03-20 17:39:00 CST        390903.333333        117834.666667  339983.333333  121141.333333
11          adservice  runtime.java_lang_MemoryPool_CollectionUsage_u...                   31              5  2022-03-20 17:39:00 CST        421128.666667        117834.666667  339983.333333  121141.333333
12          adservice  runtime.java_lang_MemoryPool_Usage_used.Surviv...                   31              5  2022-03-20 17:39:00 CST        421128.666667        117834.666667  339983.333333  121141.333333
13        adservice-2   container.node-5.container_memory_working_set_MB                   31              5  2022-03-20 17:40:00 CST           157.363281           156.296875     156.835938     156.351562
14       adservice2-0  container.node-6.container_network_transmit_MB...                   31              5  2022-03-20 17:43:00 CST             1.875155             1.188577       1.586342       1.260652
15       adservice2-0  container.node-6.container_network_transmit_pa...                   31              5  2022-03-20 17:30:00 CST           988.500000           624.500000     851.000000     645.000000
16      cartservice-1         container.node-6.container_cpu_cfs_periods                   31              5  2022-03-20 17:31:00 CST           211.500000           124.000000     203.500000     147.500000
17  checkoutservice-0       container.node-6.container_cpu_usage_seconds                   31              5  2022-03-20 17:34:00 CST             0.109887             0.029415       0.093515       0.030900
18  checkoutservice-2         container.node-5.container_memory_usage_MB                   31              5  2022-03-20 17:33:00 CST            32.187500            29.326172      32.154297      30.201172
19  checkoutservice-2  container.node-5.container_network_transmit_MB...                   31              5  2022-03-20 17:32:00 CST             2.232377             1.298756       2.212305       1.316427```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing variables if available, otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Reuse allowed_cmdb if exists, else reconstruct from problem statement
try:
    allowed_cmdb
except NameError:
    nodes = [f"node-{i}" for i in range(1, 7)]
    pods = [
        "frontend-0","frontend-1","frontend-2","frontend2-0",
        "shippingservice-0","shippingservice-1","shippingservice-2","shippingservice2-0",
        "checkoutservice-0","checkoutservice-1","checkoutservice-2","checkoutservice2-0",
        "currencyservice-0","currencyservice-1","currencyservice-2","currencyservice2-0",
        "adservice-0","adservice-1","adservice-2","adservice2-0",
        "emailservice-0","emailservice-1","emailservice-2","emailservice2-0",
        "cartservice-0","cartservice-1","cartservice-2","cartservice2-0",
        "productcatalogservice-0","productcatalogservice-1","productcatalogservice-2","productcatalogservice2-0",
        "recommendationservice-0","recommendationservice-1","recommendationservice-2","recommendationservice2-0",
        "paymentservice-0","paymentservice-1","paymentservice-2","paymentservice2-0"
    ]
    services = [
        "frontend","shippingservice","checkoutservice","currencyservice",
        "adservice","emailservice","cartservice","productcatalogservice",
        "recommendationservice","paymentservice"
    ]
    allowed_cmdb = set(nodes + pods + services)

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# 1) Restrict to allowed cmdb_id
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed_cmdb)].copy()

# 2) Compute global P95 and P5 per (cmdb_id, kpi_name) using all available rows (no time filter)
grp_all = metric_allowed.groupby(['cmdb_id','kpi_name'])['value']
thresholds_p95 = grp_all.quantile(0.95).rename('P95')
thresholds_p05 = grp_all.quantile(0.05).rename('P5')
thresholds = pd.concat([thresholds_p95, thresholds_p05], axis=1).reset_index()

# 3) Restrict to incident window
start_ts = 1647768600
end_ts = 1647770400
metric_window = metric_allowed[(metric_allowed['timestamp'] >= start_ts) & (metric_allowed['timestamp'] <= end_ts)].copy()

# If empty window, return empty DataFrame
if metric_window.shape[0] == 0:
    top10_runs = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','row_count_in_window','anomaly_count','longest_run_length',
        'longest_run_start_ts','longest_run_end_ts','run_max_value','run_min_value','P95','P5','severity_pct'
    ])
else:
    # 4) Merge thresholds into window rows and detect anomalies
    merged = metric_window.merge(thresholds, on=['cmdb_id','kpi_name'], how='left')
    merged['anomaly'] = False
    has_thresh = merged['P95'].notna() & merged['P5'].notna()
    merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh, 'value'] > merged.loc[has_thresh, 'P95']) | (merged.loc[has_thresh, 'value'] < merged.loc[has_thresh, 'P5'])

    # Compute row_count_in_window and anomaly_count per group
    group_keys = ['cmdb_id','kpi_name']
    agg_counts = merged.groupby(group_keys).agg(
        row_count_in_window=('value','size'),
        anomaly_count=('anomaly','sum')
    ).reset_index()

    # Select candidate groups: top 20 by anomaly_count desc (to ensure enough for tie-breaker resolution), then we'll compute runs and pick top 10
    candidates = agg_counts.sort_values(['anomaly_count','row_count_in_window'], ascending=[False,False]).head(20).reset_index(drop=True)

    # For each candidate compute consecutive anomaly runs
    runs_records = []
    step = 60  # timestamps are floored to minute -> consecutive if diff == 60
    for _, row in candidates.iterrows():
        cid = row['cmdb_id']
        kpi = row['kpi_name']
        df_grp = merged[(merged['cmdb_id']==cid) & (merged['kpi_name']==kpi)].sort_values('timestamp').reset_index(drop=True)
        # Extract only anomaly rows with their timestamps and values
        df_anom = df_grp[df_grp['anomaly']].loc[:, ['timestamp','value','P95','P5']].reset_index(drop=True)
        if df_anom.shape[0] == 0:
            # No runs
            runs_records.append({
                'cmdb_id': cid, 'kpi_name': kpi,
                'row_count_in_window': int(row['row_count_in_window']),
                'anomaly_count': int(row['anomaly_count']),
                'longest_run_length': 0,
                'longest_run_start_ts': None,
                'longest_run_end_ts': None,
                'run_max_value': np.nan,
                'run_min_value': np.nan,
                'P95': df_grp['P95'].dropna().iloc[0] if df_grp['P95'].notna().any() else np.nan,
                'P5': df_grp['P5'].dropna().iloc[0] if df_grp['P5'].notna().any() else np.nan,
                'severity_pct': np.nan
            })
            continue

        # iterate to find consecutive runs
        runs = []
        cur_start = df_anom.at[0,'timestamp']
        cur_end = cur_start
        cur_vals = [df_anom.at[0,'value']]
        cur_has_high = df_anom.at[0,'value'] > df_anom.at[0,'P95'] if pd.notna(df_anom.at[0,'P95']) else False
        cur_has_low = df_anom.at[0,'value'] < df_anom.at[0,'P5'] if pd.notna(df_anom.at[0,'P5']) else False

        for i in range(1, df_anom.shape[0]):
            ts = int(df_anom.at[i,'timestamp'])
            val = df_anom.at[i,'value']
            p95v = df_anom.at[i,'P95']
            p5v = df_anom.at[i,'P5']
            # check consecutive by timestamp difference
            prev_ts = int(df_anom.at[i-1,'timestamp'])
            if ts - prev_ts == step:
                # continue current run
                cur_end = ts
                cur_vals.append(val)
                if pd.notna(p95v) and val > p95v:
                    cur_has_high = True
                if pd.notna(p5v) and val < p5v:
                    cur_has_low = True
            else:
                # close current run
                runs.append({
                    'start': cur_start, 'end': cur_end, 'length': len(cur_vals),
                    'max': float(np.max(cur_vals)), 'min': float(np.min(cur_vals)),
                    'has_high': cur_has_high, 'has_low': cur_has_low
                })
                # start new run
                cur_start = ts
                cur_end = ts
                cur_vals = [val]
                cur_has_high = (pd.notna(p95v) and val > p95v)
                cur_has_low = (pd.notna(p5v) and val < p5v)
        # append last run
        runs.append({
            'start': cur_start, 'end': cur_end, 'length': len(cur_vals),
            'max': float(np.max(cur_vals)), 'min': float(np.min(cur_vals)),
            'has_high': cur_has_high, 'has_low': cur_has_low
        })

        # pick longest run (if multiple equal length, pick the earliest start)
        runs_sorted = sorted(runs, key=lambda x: (-x['length'], x['start']))
        longest = runs_sorted[0]

        # thresholds for this group (take from df_grp if present)
        P95_val = df_grp['P95'].dropna().iloc[0] if df_grp['P95'].notna().any() else np.nan
        P5_val = df_grp['P5'].dropna().iloc[0] if df_grp['P5'].notna().any() else np.nan

        # compute severity_pct: high-side vs low-side
        high_pct = np.nan
        low_pct = np.nan
        if longest['has_high'] and pd.notna(P95_val) and P95_val != 0:
            high_pct = 100.0 * (longest['max'] - P95_val) / P95_val
        if longest['has_low'] and pd.notna(P5_val) and P5_val != 0:
            low_pct = 100.0 * (P5_val - longest['min']) / P5_val

        # choose larger of two (handle NaNs)
        if np.isnan(high_pct) and np.isnan(low_pct):
            severity = np.nan
        else:
            # replace nan with -inf to allow max selection
            vals = [v for v in [high_pct, low_pct] if not pd.isna(v)]
            severity = max(vals) if vals else np.nan

        runs_records.append({
            'cmdb_id': cid, 'kpi_name': kpi,
            'row_count_in_window': int(row['row_count_in_window']),
            'anomaly_count': int(row['anomaly_count']),
            'longest_run_length': int(longest['length']),
            'longest_run_start_ts': int(longest['start']),
            'longest_run_end_ts': int(longest['end']),
            'run_max_value': float(longest['max']),
            'run_min_value': float(longest['min']),
            'P95': float(P95_val) if pd.notna(P95_val) else np.nan,
            'P5': float(P5_val) if pd.notna(P5_val) else np.nan,
            'severity_pct': float(severity) if pd.notna(severity) else np.nan
        })

    runs_df = pd.DataFrame(runs_records)

    # Final sorting: by anomaly_count desc, tie-breaker by longest_run_length desc
    top10_runs = runs_df.sort_values(['anomaly_count','longest_run_length'], ascending=[False,False]).head(10).reset_index(drop=True)

# Display final compact top-10 runs
top10_runs

```
Out[3]:
```
summary = (
"Top-10 anomaly runs within incident window (thresholds = global P95/P5 per (cmdb_id,kpi_name); anomaly = value > P95 or < P5; consecutive = contiguous minutes):\n\n"
"Overall: All top-10 pairs have 31 samples in-window and 6 anomalies each (except identical metric variants). Longest consecutive runs range 4 -> 3 minutes. Severity_pct shows how far the run exceeds P95 (or drops below P5) as a percentage.\n\n"
"1) emailservice-1 | mesh.destination.checkoutservice.emailservice... \n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=4 (start=1647769560, end=1647769740)\n"
"   - run_max=3812.5, run_min=865.0, P95=2595.0, P5=1257.5, severity_pct≈46.9% (high-side)\n\n"
"2) emailservice-1 | mesh.destination.checkoutservice.emailservice... (rate-like series)\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=4 (start=1647769560, end=1647769740)\n"
"   - run_max=0.004725, run_min=0.00105, P95=0.00315, P5=0.001575, severity_pct=50.0% (high-side)\n\n"
"3) emailservice-1 | mesh.destination.checkoutservice.emailservice... (another metric)\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=4 (start=1647769560, end=1647769740)\n"
"   - run_max=5625.0, run_min=1250.0, P95=3750.0, P5=1875.0, severity_pct=50.0% (high-side)\n\n"
"4) checkoutservice-2 | container.node-5.container_memory_working_set_MB\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769260, end=1647769380)\n"
"   - run_max=25.580078, run_min=25.462891, P95=26.90625, P5=25.765625, severity_pct≈1.17% (low-side)\n\n"
"5) frontend-1 | container.node-5.container_memory_usage_MB\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769260, end=1647769380)\n"
"   - run_max=28.839844, run_min=28.435547, P95=29.955078, P5=28.964844, severity_pct≈1.83% (low-side)\n\n"
"6) frontend-2 | mesh.destination.unknown.frontend.istio_requests... \n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769860, end=1647769980)\n"
"   - run_max=0.07855, run_min=0.015525, P95=0.068575, P5=0.022, severity_pct≈29.43% (high-side)\n\n"
"7) frontend2-0 | container.node-6.container_fs_usage_MB./dev/vda1\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769860, end=1647769980)\n"
"   - run_max=72.347656, run_min=71.865234, P95=71.539062, P5=26.222656, severity_pct≈1.13% (high-side)\n\n"
"8) node-2 | system.mem.free\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769380, end=1647769500)\n"
"   - run_max=10175.0, run_min=10136.0, P95=10110.0, P5=9754.0, severity_pct≈0.64% (high-side)\n\n"
"9) node-3 | system.load.15\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769080, end=1647769200)\n"
"   - run_max=1.05, run_min=0.99, P95=0.98, P5=0.66, severity_pct≈7.14% (high-side)\n\n"
"10) node-3 | system.mem.free\n"
"   - row_count_in_window=31, anomaly_count=6, longest_run_length=3 (start=1647769380, end=1647769500)\n"
"   - run_max=10527.0, run_min=10502.0, P95=10477.0, P5=10072.0, severity_pct≈0.48% (high-side)\n\n"
"Key takeaways:\n"
"- Several emailservice→checkoutservice mesh metrics show the longest runs (4 minutes) and the highest severities (~47–50%).\n"
"- Many other top anomalies are memory/fs/CPU/load related on frontend, checkoutservice, adservice and node-2/node-3, but their severity percentages are small (<~7%) except the mesh counts.\n"
"- Recommend immediate inspection of emailservice→checkoutservice mesh traffic and the frontend/checkoutservice memory/fs spikes around the listed start timestamps (unix seconds)."
)
summary_output = summary
summary_output

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  row_count_in_window  anomaly_count  longest_run_length  longest_run_start_ts  longest_run_end_ts  run_max_value  run_min_value           P95            P5  severity_pct
0     emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6                   4            1647769560          1647769740    3812.500000     865.000000   2595.000000   1257.500000     46.917148
1     emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6                   4            1647769560          1647769740       0.004725       0.001050      0.003150      0.001575     50.000000
2     emailservice-1  mesh.destination.checkoutservice.emailservice....                   31              6                   4            1647769560          1647769740    5625.000000    1250.000000   3750.000000   1875.000000     50.000000
3  checkoutservice-2   container.node-5.container_memory_working_set_MB                   31              6                   3            1647769260          1647769380      25.580078      25.462891     26.906250     25.765625      1.174955
4         frontend-1         container.node-5.container_memory_usage_MB                   31              6                   3            1647769260          1647769380      28.839844      28.435547     29.955078     28.964844      1.827377
5         frontend-2  mesh.destination.unknown.frontend.istio_reques...                   31              6                   3            1647769860          1647769980       0.078550       0.015525      0.068575      0.022000     29.431818
6        frontend2-0   container.node-6.container_fs_usage_MB./dev/vda1                   31              6                   3            1647769860          1647769980      72.347656      71.865234     71.539062     26.222656      1.130283
7             node-2                                    system.mem.free                   31              6                   3            1647769380          1647769500   10175.000000   10136.000000  10110.000000   9754.000000      0.642928
8             node-3                                     system.load.15                   31              6                   3            1647769080          1647769200       1.050000       0.990000      0.980000      0.660000      7.142857
9             node-3                                    system.mem.free                   31              6                   3            1647769380          1647769500   10527.000000   10502.000000  10477.000000  10072.000000      0.477236```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# reuse existing trace_df and tz if present, otherwise load/define
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric value
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, trace_name) using all rows in trace.csv
grp_all = trace_df.groupby(['cmdb_id','trace_name'])['value']
p95 = grp_all.quantile(0.95).rename('P95')
p05 = grp_all.quantile(0.05).rename('P5')
thresholds = pd.concat([p95,p05], axis=1).reset_index()

# 2) Filter trace rows to incident window
start_ts = 1647768600
end_ts = 1647770400
trace_window = trace_df[(trace_df['timestamp'] >= start_ts) & (trace_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window rows and detect anomalies
merged = trace_window.merge(thresholds, on=['cmdb_id','trace_name'], how='left')
merged['anomaly'] = False
has_thresh = merged['P95'].notna() & merged['P5'].notna()
merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh, 'value'] > merged.loc[has_thresh, 'P95']) | (merged.loc[has_thresh, 'value'] < merged.loc[has_thresh, 'P5'])

# Aggregate per (cmdb_id, trace_name)
group_keys = ['cmdb_id','trace_name']
agg = merged.groupby(group_keys).agg(
    row_count_in_window = ('value','size'),
    anomaly_count = ('anomaly','sum'),
    earliest_anomaly_ts = ('timestamp', lambda s: int(s[merged.loc[s.index,'anomaly']].min()) if merged.loc[s.index,'anomaly'].any() else np.nan),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min')
).reset_index()

# Join thresholds (P95,P5)
agg = agg.merge(thresholds, on=['cmdb_id','trace_name'], how='left')

# 4) Focus on trace series involving emailservice, checkoutservice, frontend (cmdb_id contains these substrings)
focus_keywords = ['emailservice','checkoutservice','frontend']
pattern = '|'.join(focus_keywords)
agg_focus = agg[agg['cmdb_id'].str.contains(pattern, na=False)].copy()

# Prepare sorting: anomaly_count desc, tie-breaker earliest_anomaly_ts asc (NaN -> large value)
agg_focus['earliest_for_sort'] = agg_focus['earliest_anomaly_ts'].fillna(10**12)
top20_trace_anomalies = agg_focus.sort_values(['anomaly_count','earliest_for_sort'], ascending=[False,True]).head(20)

# Select and order required columns
top20_trace_anomalies = top20_trace_anomalies[[
    'cmdb_id','trace_name','row_count_in_window','anomaly_count','earliest_anomaly_ts',
    'max_value_in_window','min_value_in_window','P95','P5'
]].reset_index(drop=True)

# Display compact top 20
top20_trace_anomalies

```
Out[4]:
```
Summary of trace anomaly scan (global P95/P5 per (cmdb_id, trace_name); incident window = timestamps 1647768600…1647770400):

What I found
- Top anomalous trace series are dominated by checkoutservice (checkoutservice-0/1/2 and checkoutservice2-0) calling downstream services (productcatalogservice, currencyservice, emailservice, shippingservice) and several checkoutservice self/from_root series.
- All top 20 returned rows have anomaly_count = 4 (i.e., 4 points in-window breaching P95 or P5) with row_count_in_window typically 22–30.
- Earliest anomaly timestamps for the top results are around 1.647769e+09 (i.e., within the incident window).
- Examples of top trace anomalies (from the displayed top 20):
  - checkoutservice-0 → productcatalogservice-0: duration_mean and duration_p95 (row_count 27, anomaly_count 4).
  - checkoutservice-1 → productcatalogservice-2: duration_p95 (row_count 25, anomaly_count 4).
  - checkoutservice-2: trace.self.duration_mean and many to productcatalogservice-1/2, to_currencyservice-*, to_emailservice-1, to_shippingservice-2 (row_count 22–30, anomaly_count 4).
  - checkoutservice2-0: traces from frontend2-0, from_root, self.duration_p95, to_productcatalogservice2-0 (row_count 30, anomaly_count 4).

Interpretation / implications
- The pattern points to checkoutservice (and its variants) experiencing anomalous latencies or distribution shifts in calls to productcatalogservice, currencyservice, emailservice and shippingservice during the incident window.
- These anomalies are consistent across multiple trace metrics (mean and p95), suggesting end-to-end latency increases rather than a single noisy metric.

Recommended next steps
- Inspect checkoutservice traces in detail around the earliest anomaly timestamps (around 1647769xxx) to see call chains and which downstream calls add latency.
- Investigate the downstream services identified (productcatalogservice, currencyservice, emailservice, shippingservice) for resource pressure, errors, or slow responses at the same times.
- Correlate with metric and log data (CPU/memory, network, and pod logs) for the involved checkoutservice and downstream pods to find root cause (resource, network, or service-level issue).

If you want, I can produce the compact table again with exact numeric P95/P5 and earliest_anomaly_ts values for any subset of these trace rows.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  row_count_in_window  anomaly_count  earliest_anomaly_ts  max_value_in_window  min_value_in_window       P95        P5
0    checkoutservice-0   trace.to_productcatalogservice-0.duration_mean                   27              4         1.647769e+09             0.006559             0.004431  0.005986  0.004497
1    checkoutservice-0    trace.to_productcatalogservice-0.duration_p95                   27              4         1.647769e+09             0.006743             0.004431  0.006239  0.004507
2    checkoutservice-1    trace.to_productcatalogservice-2.duration_p95                   25              4         1.647769e+09             0.006080             0.004204  0.005810  0.004634
3    checkoutservice-2                         trace.self.duration_mean                   30              4         1.647769e+09             0.008948             0.004083  0.008573  0.004286
4    checkoutservice-2          trace.to_currencyservice-0.duration_p95                   30              4         1.647769e+09             0.000690             0.000111  0.000144  0.000112
5    checkoutservice-2         trace.to_currencyservice-1.duration_mean                   30              4         1.647769e+09             0.000198             0.000082  0.000141  0.000107
6    checkoutservice-2          trace.to_currencyservice-1.duration_p95                   30              4         1.647769e+09             0.000198             0.000082  0.000145  0.000115
7    checkoutservice-2            trace.to_emailservice-1.duration_mean                   22              4         1.647769e+09             0.000358             0.000234  0.000272  0.000235
8    checkoutservice-2             trace.to_emailservice-1.duration_p95                   22              4         1.647769e+09             0.000358             0.000234  0.000272  0.000235
9    checkoutservice-2   trace.to_productcatalogservice-1.duration_mean                   29              4         1.647769e+09             0.005707             0.003935  0.005554  0.004145
10   checkoutservice-2    trace.to_productcatalogservice-1.duration_p95                   29              4         1.647769e+09             0.006550             0.003935  0.005812  0.004235
11   checkoutservice-2   trace.to_productcatalogservice-2.duration_mean                   29              4         1.647769e+09             0.006265             0.004441  0.005915  0.004595
12   checkoutservice-2    trace.to_productcatalogservice-2.duration_p95                   29              4         1.647769e+09             0.007306             0.004441  0.005915  0.004720
13   checkoutservice-2         trace.to_shippingservice-2.duration_mean                   29              4         1.647769e+09             0.000184             0.000057  0.000078  0.000057
14   checkoutservice-2          trace.to_shippingservice-2.duration_p95                   29              4         1.647769e+09             0.000297             0.000057  0.000080  0.000057
15  checkoutservice2-0              trace.from_frontend2-0.duration_p95                   30              4         1.647769e+09             0.124145             0.043989  0.103962  0.064331
16  checkoutservice2-0                    trace.from_root.duration_mean                   30              4         1.647769e+09             0.005341             0.002577  0.004176  0.002609
17  checkoutservice2-0                     trace.from_root.duration_p95                   30              4         1.647769e+09             0.010103             0.003099  0.003862  0.003172
18  checkoutservice2-0                          trace.self.duration_p95                   30              4         1.647769e+09             0.025671             0.008053  0.021958  0.008117
19  checkoutservice2-0  trace.to_productcatalogservice2-0.duration_mean                   30              4         1.647769e+09             0.005558             0.004690  0.005248  0.004809```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse existing log_df if present
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# Ensure numeric values
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

# 1) Compute global P95 and P5 per (cmdb_id, log_name) using all rows
grp_all = log_df.groupby(['cmdb_id','log_name'])['value']
p95 = grp_all.quantile(0.95).rename('P95')
p05 = grp_all.quantile(0.05).rename('P5')
thresholds_log = pd.concat([p95,p05], axis=1).reset_index()

# 2) Filter log rows to incident window
start_ts = 1647768600
end_ts = 1647770400
log_window = log_df[(log_df['timestamp'] >= start_ts) & (log_df['timestamp'] <= end_ts)].copy()

# 3) Merge thresholds into window rows and detect anomalies
merged_log = log_window.merge(thresholds_log, on=['cmdb_id','log_name'], how='left')
merged_log['anomaly'] = False
has_thresh = merged_log['P95'].notna() & merged_log['P5'].notna()
merged_log.loc[has_thresh, 'anomaly'] = (merged_log.loc[has_thresh, 'value'] > merged_log.loc[has_thresh, 'P95']) | (merged_log.loc[has_thresh, 'value'] < merged_log.loc[has_thresh, 'P5'])

# Aggregations
group_keys = ['cmdb_id','log_name']
agg_counts = merged_log.groupby(group_keys).agg(
    row_count_in_window = ('value','size'),
    anomaly_count = ('anomaly','sum'),
    max_value_in_window = ('value','max'),
    min_value_in_window = ('value','min')
).reset_index()

# earliest anomaly timestamp per group (if any)
earliest_anom = merged_log[merged_log['anomaly']].groupby(group_keys)['timestamp'].min().rename('earliest_anomaly_ts').reset_index()
agg = agg_counts.merge(earliest_anom, on=group_keys, how='left')

# Join thresholds
agg = agg.merge(thresholds_log, on=group_keys, how='left')

# Keep columns ordered as requested
agg = agg[['cmdb_id','log_name','row_count_in_window','anomaly_count','earliest_anomaly_ts','max_value_in_window','min_value_in_window','P95','P5']]

# 4) Sort by anomaly_count desc, tie-breaker earliest_anomaly_ts asc (NaN treated as large)
agg['earliest_for_sort'] = agg['earliest_anomaly_ts'].fillna(10**12)
top20 = agg.sort_values(['anomaly_count','earliest_for_sort'], ascending=[False,True]).head(20).reset_index(drop=True)
top20 = top20.drop(columns=['earliest_for_sort'])

# 5) Ensure emailservice-1 and checkoutservice-* rows included if present but not in top20
# Build candidate pairs present in full agg
candidates_mask = agg['cmdb_id'].str.match(r'^(checkoutservice(?:$|-)|emailservice-1$)', na=False)
candidates = agg[candidates_mask].copy()

# Identify (cmdb_id,log_name) pairs not already in top20
top20_keys = set(zip(top20['cmdb_id'], top20['log_name']))
extras = []
for _, r in candidates.iterrows():
    key = (r['cmdb_id'], r['log_name'])
    if key not in top20_keys:
        extras.append(r)

# Append up to (22 - len(top20)) extra rows
max_total = 22
slots = max_total - len(top20)
if slots > 0 and len(extras) > 0:
    extras_df = pd.DataFrame(extras).head(slots).reset_index(drop=True)
    final_out = pd.concat([top20, extras_df], ignore_index=True)
else:
    final_out = top20

# Keep output compact: show up to 22 rows as requested
log_top_output = final_out.reset_index(drop=True)
log_top_output

```
Out[5]:
```
summary = (
    "Log anomaly summary (global P95/P5 per cmdb_id+log_name; incident window 1647768600–1647770400):\n\n"
    "Key findings:\n"
    "- Top anomalies are primarily in log.row_count metrics for backend services.\n"
    "- currencyservice-0/1/2: each has 31 samples and 4 anomalies. Peak row_counts: 585 / 593 / 595 with P95 ≈ 565.5 / 570.5 / 575.5.\n"
    "- cartservice-2: 31 samples, 4 anomalies, max row_count 627 (P95 ≈ 613).\n"
    "- shippingservice-0: 30 samples, 4 anomalies, max row_count 87 (P95 ≈ 77.55).\n"
    "- Several other services (currencyservice*, cartservice*, shippingservice, adservice, etc.) also show 3–4 anomalies in log.row_count.\n"
    "- checkoutservice-0 has 30 samples and 3 anomalies in log.row_count; checkoutservice-0 and checkoutservice-1 show log.error_count = 0 (no error_count anomalies).\n\n"
    "Implication & recommended next steps:\n"
    "- The pattern points to unusual request counts (row_count) on currencyservice and cartservice (and shippingservice) during the incident window. This could indicate traffic spikes/drops or instrumentation issues.\n"
    "- Inspect traces and metrics for these services (CPU, memory, network) around the earliest anomaly timestamps and correlate with checkoutservice traces (since checkoutservice appears in related trace anomalies).\n"
    "- Check application logs for currencyservice/cartservice for errors or rate/throughput changes; verify any recent deployments or load balancer/ingress behavior.\n"
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  row_count_in_window  anomaly_count  earliest_anomaly_ts  max_value_in_window  min_value_in_window     P95      P5  earliest_for_sort
0   currencyservice-0    log.row_count                   31              4         1.647769e+09                585.0                  5.0  565.50  392.50                NaN
1   currencyservice-1    log.row_count                   31              4         1.647769e+09                593.0                  1.0  570.50  394.50                NaN
2   currencyservice-2    log.row_count                   31              4         1.647769e+09                595.0                  4.0  575.50  399.50                NaN
3   shippingservice-0    log.row_count                   30              4         1.647769e+09                 87.0                 44.0   77.55   46.45                NaN
4       cartservice-2    log.row_count                   31              4         1.647769e+09                627.0                 19.0  613.00  506.50                NaN
..                ...              ...                  ...            ...                  ...                  ...                  ...     ...     ...                ...
17        adservice-0    log.row_count                   30              3         1.647769e+09                 98.0                 68.0   92.00   70.90                NaN
18        adservice-2    log.row_count                   30              3         1.647769e+09                 88.0                 58.0   80.00   58.90                NaN
19  checkoutservice-0    log.row_count                   30              3         1.647769e+09                 68.0                 14.0   66.20   17.00                NaN
20  checkoutservice-0  log.error_count                   30              0                  NaN                  0.0                  0.0    0.00    0.00       1.000000e+12
21  checkoutservice-1  log.error_count                   30              0                  NaN                  0.0                  0.0    0.00    0.00       1.000000e+12

[22 rows x 10 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

tz = pytz.timezone('Asia/Shanghai')

# reuse metric_df if present
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Focus on emailservice-1
cmdb_target = 'emailservice-1'
df_email = metric_df[metric_df['cmdb_id'] == cmdb_target].copy()

# If no data for target, produce empty dataframe with requested columns
cols = ['kpi_name','row_count_in_window','anomaly_count','longest_run_length','longest_run_start_ts',
        'earliest_anomaly_ts','run_max_value','run_min_value','P95','P5','severity_pct']
if df_email.shape[0] == 0:
    email_top20 = pd.DataFrame(columns=cols)
else:
    # 1) Global P95 and P5 per kpi_name using all available rows for emailservice-1
    grp_all = df_email.groupby('kpi_name')['value']
    p95 = grp_all.quantile(0.95).rename('P95')
    p05 = grp_all.quantile(0.05).rename('P5')
    thresholds = pd.concat([p95,p05], axis=1).reset_index()

    # 2) Filter to incident window
    start_ts = 1647768600
    end_ts = 1647770400
    df_win = df_email[(df_email['timestamp'] >= start_ts) & (df_email['timestamp'] <= end_ts)].copy()

    if df_win.shape[0] == 0:
        email_top20 = pd.DataFrame(columns=cols)
    else:
        # 3) Merge thresholds and mark anomalies
        merged = df_win.merge(thresholds, on='kpi_name', how='left')
        merged['anomaly'] = False
        has_thresh = merged['P95'].notna() & merged['P5'].notna()
        merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh, 'value'] > merged.loc[has_thresh, 'P95']) | (merged.loc[has_thresh, 'value'] < merged.loc[has_thresh, 'P5'])

        # Aggregations per kpi_name
        agg = merged.groupby('kpi_name').agg(
            row_count_in_window = ('value','size'),
            anomaly_count = ('anomaly','sum'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()

        earliest = merged[merged['anomaly']].groupby('kpi_name')['timestamp'].min().rename('earliest_anomaly_ts').reset_index()
        agg = agg.merge(earliest, on='kpi_name', how='left')
        agg = agg.merge(thresholds, on='kpi_name', how='left')

        # 4) Identify consecutive anomaly runs and compute longest run per kpi_name
        runs_records = []
        step = 60  # consecutive minute granularity
        for kpi, grp in merged.groupby('kpi_name'):
            grp_sorted = grp.sort_values('timestamp').reset_index(drop=True)
            anom_rows = grp_sorted[grp_sorted['anomaly']].loc[:, ['timestamp','value','P95','P5']].reset_index(drop=True)
            if anom_rows.shape[0] == 0:
                runs_records.append({
                    'kpi_name': kpi,
                    'longest_run_length': 0,
                    'longest_run_start_ts': None,
                    'run_max_value': np.nan,
                    'run_min_value': np.nan,
                    'run_has_high': False,
                    'run_has_low': False
                })
                continue

            # scan for consecutive runs
            runs = []
            cur_start = int(anom_rows.at[0,'timestamp'])
            cur_end = cur_start
            vals = [float(anom_rows.at[0,'value'])]
            has_high = (pd.notna(anom_rows.at[0,'P95']) and anom_rows.at[0,'value'] > anom_rows.at[0,'P95'])
            has_low = (pd.notna(anom_rows.at[0,'P5']) and anom_rows.at[0,'value'] < anom_rows.at[0,'P5'])
            for i in range(1, anom_rows.shape[0]):
                ts = int(anom_rows.at[i,'timestamp'])
                val = float(anom_rows.at[i,'value'])
                p95v = anom_rows.at[i,'P95']
                p5v = anom_rows.at[i,'P5']
                prev_ts = int(anom_rows.at[i-1,'timestamp'])
                if ts - prev_ts == step:
                    cur_end = ts
                    vals.append(val)
                    if pd.notna(p95v) and val > p95v:
                        has_high = True
                    if pd.notna(p5v) and val < p5v:
                        has_low = True
                else:
                    runs.append({
                        'start': cur_start, 'end': cur_end, 'length': len(vals),
                        'max': max(vals), 'min': min(vals),
                        'has_high': has_high, 'has_low': has_low
                    })
                    # reset
                    cur_start = ts
                    cur_end = ts
                    vals = [val]
                    has_high = (pd.notna(p95v) and val > p95v)
                    has_low = (pd.notna(p5v) and val < p5v)
            # append last
            runs.append({
                'start': cur_start, 'end': cur_end, 'length': len(vals),
                'max': max(vals), 'min': min(vals),
                'has_high': has_high, 'has_low': has_low
            })

            # select longest run (tie -> earliest start)
            runs_sorted = sorted(runs, key=lambda x: (-x['length'], x['start']))
            longest = runs_sorted[0]
            runs_records.append({
                'kpi_name': kpi,
                'longest_run_length': int(longest['length']),
                'longest_run_start_ts': int(longest['start']),
                'run_max_value': float(longest['max']),
                'run_min_value': float(longest['min']),
                'run_has_high': bool(longest['has_high']),
                'run_has_low': bool(longest['has_low'])
            })

        runs_df = pd.DataFrame(runs_records)

        # merge runs info into agg
        agg_full = agg.merge(runs_df, on='kpi_name', how='left')

        # compute severity_pct per definition
        def compute_severity(row):
            P95v = row['P95']
            P5v = row['P5']
            maxv = row['run_max_value']
            minv = row['run_min_value']
            high_pct = np.nan
            low_pct = np.nan
            if row['run_has_high'] and pd.notna(P95v) and P95v != 0:
                high_pct = 100.0 * (maxv - P95v) / P95v
            if row['run_has_low'] and pd.notna(P5v) and P5v != 0:
                low_pct = 100.0 * (P5v - minv) / P5v
            if pd.isna(high_pct) and pd.isna(low_pct):
                return np.nan
            candidates = [v for v in [high_pct, low_pct] if not pd.isna(v)]
            return float(max(candidates)) if candidates else np.nan

        agg_full['severity_pct'] = agg_full.apply(compute_severity, axis=1)

        # Prepare final columns and sorting
        final = agg_full.rename(columns={
            'max_value_in_window':'run_max_value_in_window_tmp'  # not used
        })  # keep names clear
        # Select requested output columns
        email_top20 = final[['kpi_name','row_count_in_window','anomaly_count','longest_run_length',
                             'longest_run_start_ts','earliest_anomaly_ts','run_max_value','run_min_value','P95','P5','severity_pct']].copy()

        # Sort by anomaly_count desc, then longest_run_length desc, and limit to top 20
        email_top20 = email_top20.sort_values(['anomaly_count','longest_run_length'], ascending=[False,False]).head(20).reset_index(drop=True)

# Display compact top 20 for emailservice-1
email_top20

```
Out[6]:
```
Summary of metric anomalies for emailservice-1 (incident window 1647768600–1647770400):

Key findings
- Multiple mesh destination metrics toward checkoutservice from emailservice-1 are the most anomalous:
  - mesh.destination.checkoutservice.emailservice... (three distinct metrics) — each: 31 samples in-window, 6 anomalies, longest consecutive run = 4 minutes (longest_run_start_ts shown ≈ 1.64777e+09), severity_pct ≈ 46.9% and 50.0% (large high-side breaches).
  - Additional mesh/source jaeger-collector series also show strong anomalies (4 anomalies, longest run 4, severity ≈ 50%).

- Resource and network metrics on the same pod also show anomalies (fewer or lower severity):
  - container.node-6.container_cpu_usage_seconds — 31 samples, 4 anomalies, longest run 2, severity ≈ 10.3%.
  - container.node-6 network receive/transmit metrics — 31 samples, 2–3 anomalies, severities in the single-/two-digit percent range.
  - Some fs/last_seen series show small anomalies or timestamp jumps.

- Some kpis show zero anomalies (e.g., container_cpu_cfs_throttled_pct).

Interpretation
- The strongest signals are mesh (service-to-service) metrics from emailservice-1 to checkoutservice: multi-minute consecutive breaches with high severity (≈47–50%). That suggests emailservice-1 experienced abnormal outbound request volume/latency toward checkoutservice during the incident window.
- CPU and network metrics on the same node/container exhibit smaller but notable anomalies that could be related (resource pressure or network behavior).

Recommended next steps
1. Inspect emailservice-1 traces and request logs around the longest_run_start_ts (~1.64777e+09) to see why calls to checkoutservice spiked or slowed (e.g., retry storms, increased payloads, timeouts).
2. Check Jaeger/collector traffic and mesh proxy logs for high request/latency patterns (mesh.source/mesh.destination anomalies).
3. Correlate with node-6 container resource metrics (CPU, network, disk) to determine whether resource constraints contributed.
4. If needed, throttle or investigate upstream traffic sources that cause the increased mesh load.

If you want, I can export the compact table of the 20 kpi rows shown above with exact unix timestamps and numeric values for a focused inspection.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  row_count_in_window  anomaly_count  longest_run_length  longest_run_start_ts  earliest_anomaly_ts  run_max_value  run_min_value           P95            P5  severity_pct
0   mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09   3.812500e+03   8.650000e+02  2.595000e+03  1.257500e+03  4.691715e+01
1   mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09   4.725000e-03   1.050000e-03  3.150000e-03  1.575000e-03  5.000000e+01
2   mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09   5.625000e+03   1.250000e+03  3.750000e+03  1.875000e+03  5.000000e+01
3   mesh.source.emailservice.jaeger-collector.isti...                   31              4                   4          1.647770e+09         1.647770e+09   4.500000e+00   1.000000e+00  3.000000e+00  1.500000e+00  5.000000e+01
4   mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09   4.500000e+00   1.000000e+00  3.000000e+00  1.500000e+00  5.000000e+01
5   mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09   4.500000e+00   1.000000e+00  3.000000e+00  1.500000e+00  5.000000e+01
6   mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09   4.500000e+00   1.000000e+00  3.000000e+00  1.500000e+00  5.000000e+01
7        container.node-6.container_cpu_usage_seconds                   31              4                   2          1.647770e+09         1.647769e+09   4.947371e-02   2.877920e-02  4.747518e-02  3.208284e-02  1.029721e+01
8          container.node-6.container_cpu_cfs_periods                   31              4                   1          1.647769e+09         1.647769e+09   8.750000e+01   8.750000e+01  8.150000e+01  5.500000e+01  7.361963e+00
9                container.node-6.container_last_seen                   31              3                   3          1.647770e+09         1.647770e+09   1.647770e+09   1.647770e+09  1.647770e+09  1.647767e+09  1.092385e-05
10  container.node-6.container_network_receive_MB....                   31              3                   2          1.647769e+09         1.647769e+09   3.358808e-01   3.310361e-01  3.098450e-01  2.209473e-02  8.402843e+00
11  container.node-6.container_network_receive_pac...                   31              3                   2          1.647769e+09         1.647769e+09   3.925000e+02   3.520000e+02  3.440000e+02  2.580000e+02  1.409884e+01
12  container.node-6.container_network_transmit_pa...                   31              3                   2          1.647769e+09         1.647769e+09   2.815000e+02   2.470000e+02  2.390000e+02  1.660000e+02  1.778243e+01
13  mesh.source.emailservice.jaeger-collector.isti...                   31              3                   2          1.647770e+09         1.647769e+09   8.775000e+03   1.950000e+03  5.850000e+03  2.250000e+03  5.000000e+01
14   container.node-6.container_fs_usage_MB./dev/vda1                   31              2                   2          1.647770e+09         1.647770e+09   3.669922e+01   3.669727e+01  3.669531e+01  3.666406e+01  1.064509e-02
15      container.node-6.container_cpu_system_seconds                   31              2                   1          1.647769e+09         1.647769e+09   1.500000e-02   1.500000e-02  1.500000e-02  0.000000e+00  1.893277e-10
16  container.node-6.container_network_transmit_MB...                   31              2                   1          1.647770e+09         1.647770e+09   4.488363e-01   4.488363e-01  4.485917e-01  1.668372e-01  5.453009e-02
17        container.node-6.container_cpu_user_seconds                   31              1                   1          1.647769e+09         1.647769e+09   3.000000e-02   3.000000e-02  2.500000e-02  1.500000e-02  2.000000e+01
18  mesh.source.emailservice.jaeger-collector.isti...                   31              1                   1          1.647770e+09         1.647770e+09   4.477500e+02   4.477500e+02  2.985000e+02  9.950000e+01  5.000000e+01
19  container.node-6.container_cpu_cfs_throttled_p...                   31              0                   0                   NaN                  NaN            NaN            NaN  0.000000e+00  0.000000e+00           NaN```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df and tz if present, otherwise load/define
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

# Ensure numeric value column
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Target cmdb_id
cmdb_target = 'emailservice-1'

# 1) Global thresholds (P95, P5) per kpi_name using all available rows for the target
df_email_all = metric_df[metric_df['cmdb_id'] == cmdb_target].copy()
if df_email_all.shape[0] == 0:
    email_top10_severity = pd.DataFrame(columns=[
        'kpi_name','row_count_in_window','anomaly_count','longest_run_length','longest_run_start_ts',
        'earliest_anomaly_ts','run_max_value','run_min_value','P95','P5','severity_pct','severity_over_50pct'
    ])
else:
    grp_all = df_email_all.groupby('kpi_name')['value']
    thresholds_p95 = grp_all.quantile(0.95).rename('P95')
    thresholds_p05 = grp_all.quantile(0.05).rename('P5')
    thresholds = pd.concat([thresholds_p95, thresholds_p05], axis=1).reset_index()

    # 2) Filter to incident window
    start_ts = 1647768600
    end_ts = 1647770400
    df_win = df_email_all[(df_email_all['timestamp'] >= start_ts) & (df_email_all['timestamp'] <= end_ts)].copy()

    if df_win.shape[0] == 0:
        email_top10_severity = pd.DataFrame(columns=[
            'kpi_name','row_count_in_window','anomaly_count','longest_run_length','longest_run_start_ts',
            'earliest_anomaly_ts','run_max_value','run_min_value','P95','P5','severity_pct','severity_over_50pct'
        ])
    else:
        # 3) Merge thresholds and detect anomalies
        merged = df_win.merge(thresholds, on='kpi_name', how='left')
        merged['anomaly'] = False
        has_thresh = merged['P95'].notna() & merged['P5'].notna()
        merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh, 'value'] > merged.loc[has_thresh, 'P95']) | (merged.loc[has_thresh, 'value'] < merged.loc[has_thresh, 'P5'])

        # Aggregations per kpi_name
        agg_basic = merged.groupby('kpi_name').agg(
            row_count_in_window = ('value','size'),
            anomaly_count = ('anomaly','sum'),
            max_value_in_window = ('value','max'),
            min_value_in_window = ('value','min')
        ).reset_index()
        earliest = merged[merged['anomaly']].groupby('kpi_name')['timestamp'].min().rename('earliest_anomaly_ts').reset_index()
        agg = agg_basic.merge(earliest, on='kpi_name', how='left')
        agg = agg.merge(thresholds, on='kpi_name', how='left')

        # 4) Identify consecutive anomaly runs and produce longest run info per kpi_name
        runs_list = []
        step = 60  # seconds between consecutive timestamps (per-minute)
        for kpi, grp in merged.groupby('kpi_name'):
            grp_sorted = grp.sort_values('timestamp').reset_index(drop=True)
            anoms = grp_sorted[grp_sorted['anomaly']][['timestamp','value','P95','P5']].reset_index(drop=True)
            if anoms.shape[0] == 0:
                runs_list.append({
                    'kpi_name': kpi,
                    'longest_run_length': 0,
                    'longest_run_start_ts': np.nan,
                    'run_max_value': np.nan,
                    'run_min_value': np.nan,
                    'run_has_high': False,
                    'run_has_low': False
                })
                continue

            # scan for consecutive runs
            runs = []
            cur_start = int(anoms.at[0,'timestamp'])
            cur_end = cur_start
            vals = [float(anoms.at[0,'value'])]
            has_high = (pd.notna(anoms.at[0,'P95']) and anoms.at[0,'value'] > anoms.at[0,'P95'])
            has_low = (pd.notna(anoms.at[0,'P5']) and anoms.at[0,'value'] < anoms.at[0,'P5'])
            for i in range(1, anoms.shape[0]):
                ts = int(anoms.at[i,'timestamp'])
                val = float(anoms.at[i,'value'])
                p95v = anoms.at[i,'P95']
                p5v = anoms.at[i,'P5']
                prev_ts = int(anoms.at[i-1,'timestamp'])
                if ts - prev_ts == step:
                    cur_end = ts
                    vals.append(val)
                    if pd.notna(p95v) and val > p95v:
                        has_high = True
                    if pd.notna(p5v) and val < p5v:
                        has_low = True
                else:
                    runs.append({
                        'start': cur_start, 'end': cur_end, 'length': len(vals),
                        'max': max(vals), 'min': min(vals),
                        'has_high': has_high, 'has_low': has_low
                    })
                    cur_start = ts
                    cur_end = ts
                    vals = [val]
                    has_high = (pd.notna(p95v) and val > p95v)
                    has_low = (pd.notna(p5v) and val < p5v)
            runs.append({
                'start': cur_start, 'end': cur_end, 'length': len(vals),
                'max': max(vals), 'min': min(vals),
                'has_high': has_high, 'has_low': has_low
            })

            # choose longest run (tie-breaker earliest start)
            runs_sorted = sorted(runs, key=lambda r: (-r['length'], r['start']))
            longest = runs_sorted[0]
            runs_list.append({
                'kpi_name': kpi,
                'longest_run_length': int(longest['length']),
                'longest_run_start_ts': int(longest['start']),
                'run_max_value': float(longest['max']),
                'run_min_value': float(longest['min']),
                'run_has_high': bool(longest['has_high']),
                'run_has_low': bool(longest['has_low'])
            })

        runs_df = pd.DataFrame(runs_list)

        # combine agg and runs info
        result = agg.merge(runs_df, on='kpi_name', how='left')

        # 5) compute severity_pct per longest run
        def severity_calc(r):
            P95v = r['P95']
            P5v = r['P5']
            maxv = r['run_max_value']
            minv = r['run_min_value']
            high_pct = np.nan
            low_pct = np.nan
            if r.get('run_has_high', False) and pd.notna(P95v) and P95v != 0:
                high_pct = 100.0 * (maxv - P95v) / P95v
            if r.get('run_has_low', False) and pd.notna(P5v) and P5v != 0:
                low_pct = 100.0 * (P5v - minv) / P5v
            if pd.isna(high_pct) and pd.isna(low_pct):
                return np.nan
            candidates = [v for v in [high_pct, low_pct] if not pd.isna(v)]
            return float(max(candidates)) if candidates else np.nan

        result['severity_pct'] = result.apply(severity_calc, axis=1)
        result['severity_over_50pct'] = result['severity_pct'].apply(lambda x: bool(x > 50) if pd.notna(x) else False)

        # select and order requested columns
        email_top10_severity = result[[
            'kpi_name','row_count_in_window','anomaly_count','longest_run_length','longest_run_start_ts',
            'earliest_anomaly_ts','run_max_value','run_min_value','P95','P5','severity_pct','severity_over_50pct'
        ]].copy()

        # Sort by severity_pct desc (NaNs last), tie-breaker anomaly_count desc, limit 10
        email_top10_severity = email_top10_severity.sort_values(
            by=['severity_pct','anomaly_count'], ascending=[False,False], na_position='last'
        ).head(10).reset_index(drop=True)

# Final compact output variable
email_top10_severity

```
Out[7]:
```
Summary — emailservice-1 metric anomaly scan (global P95/P5 per KPI; incident window = 1647768600…1647770400):

What stands out
- The strongest anomalies are mesh metrics from emailservice-1 toward checkoutservice:
  - Several mesh.destination.checkoutservice.* KPIs have high severity (≈50% over P95) and the largest anomaly impact. Example: one KPI had anomaly_count=6, longest consecutive run=4 minutes, run_max=5625, P95=3750 (severity_pct = 50.0%).
  - Another mesh destination series (small fractional rates) also shows anomaly_count=6, longest run=4, severity_pct = 50.0%.
  - A related mesh.destination series shows severity ≈46.9% with anomaly_count=6 and longest run=4.
- Jaeger/collector-related mesh.source series also show notable anomalies (severity ≈50%, runs up to 4 minutes).
- Container-level resource metrics on node-6 (CPU, network, fs) show some anomalies but with lower severity (single- or low-double-digit percent).
- severity_over_50pct is False for all returned KPIs (no runs exceed 50% by more than 50%).

Interpretation
- Emailservice-1 experienced multi-minute consecutive breaches in outbound mesh metrics toward checkoutservice (both count/volume-like and rate-like KPIs). That suggests abnormal outbound traffic/latency or request-volume behavior from emailservice-1 to checkoutservice during the incident window.
- Jaeger/collector traffic anomalies suggest increased tracing/collector load in the same timeframe.
- Resource metrics on the same container/node show smaller deviations that may be related but are less severe.

Actionable next steps
1. Inspect emailservice-1 outbound logs and traces around the earliest anomaly timestamps to determine why calls to checkoutservice spiked or slowed (retries, batching, increased traffic).
2. Check Jaeger/collector and mesh proxy logs for request amplification or instrumentation issues.
3. Correlate with node-6 resource metrics and pod logs to see if resource pressure or network issues contributed.
4. If confirmed load spike or retry storm, consider throttling upstream or mitigating retries while investigating.

If you want, I can produce the exact top-10 KPI table again with the raw unix timestamps and numeric values for a focused inspection.

The original code execution output of IPython Kernel is also provided below for reference:

kpi_name  row_count_in_window  anomaly_count  longest_run_length  longest_run_start_ts  earliest_anomaly_ts  run_max_value  run_min_value         P95           P5  severity_pct  severity_over_50pct
0  mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09    5625.000000     1250.00000  3750.00000  1875.000000     50.000000                False
1  mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09       4.500000        1.00000     3.00000     1.500000     50.000000                False
2  mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09       4.500000        1.00000     3.00000     1.500000     50.000000                False
3  mesh.destination.checkoutservice.emailservice....                   31              4                   3          1.647770e+09         1.647769e+09       4.500000        1.00000     3.00000     1.500000     50.000000                False
4  mesh.source.emailservice.jaeger-collector.isti...                   31              4                   4          1.647770e+09         1.647770e+09       4.500000        1.00000     3.00000     1.500000     50.000000                False
5  mesh.source.emailservice.jaeger-collector.isti...                   31              3                   2          1.647770e+09         1.647769e+09    8775.000000     1950.00000  5850.00000  2250.000000     50.000000                False
6  mesh.source.emailservice.jaeger-collector.isti...                   31              1                   1          1.647770e+09         1.647770e+09     447.750000      447.75000   298.50000    99.500000     50.000000                False
7  mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09       0.004725        0.00105     0.00315     0.001575     50.000000                False
8  mesh.destination.checkoutservice.emailservice....                   31              6                   4          1.647770e+09         1.647769e+09    3812.500000      865.00000  2595.00000  1257.500000     46.917148                False
9        container.node-6.container_cpu_user_seconds                   31              1                   1          1.647769e+09         1.647769e+09       0.030000        0.03000     0.02500     0.015000     20.000000                False```
```

In [ ]:
# In[8]:

import pandas as pd
import numpy as np
import pytz

# reuse existing metric_df if available
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")

tz = pytz.timezone('Asia/Shanghai')

# Target cmdb_id
cmdb_target = 'emailservice-1'

# 1) Global P95 and P5 per kpi_name using all available rows for emailservice-1
df_email_all = metric_df[metric_df['cmdb_id'] == cmdb_target].copy()
if df_email_all.shape[0] == 0:
    earliest_anomalies_df = pd.DataFrame(columns=['timestamp','kpi_name','value','P95','P5','breach_side','severity_pct'])
else:
    df_email_all['value'] = pd.to_numeric(df_email_all['value'], errors='coerce')
    thresholds = df_email_all.groupby('kpi_name')['value'].quantile([0.95,0.05]).unstack(level=1)
    thresholds.columns = ['P95','P5']
    thresholds = thresholds.reset_index()

    # 2) Filter to incident window
    start_ts = 1647768600
    end_ts = 1647770400
    df_win = df_email_all[(df_email_all['timestamp'] >= start_ts) & (df_email_all['timestamp'] <= end_ts)].copy()

    if df_win.shape[0] == 0:
        earliest_anomalies_df = pd.DataFrame(columns=['timestamp','kpi_name','value','P95','P5','breach_side','severity_pct'])
    else:
        # 3) Merge thresholds and mark anomalies
        merged = df_win.merge(thresholds, on='kpi_name', how='left')
        merged['anomaly'] = False
        has_thresh = merged['P95'].notna() & merged['P5'].notna()
        merged.loc[has_thresh, 'anomaly'] = (merged.loc[has_thresh,'value'] > merged.loc[has_thresh,'P95']) | (merged.loc[has_thresh,'value'] < merged.loc[has_thresh,'P5'])

        # find earliest anomaly timestamp across all kpi_name
        if not merged['anomaly'].any():
            earliest_anomalies_df = pd.DataFrame(columns=['timestamp','kpi_name','value','P95','P5','breach_side','severity_pct'])
        else:
            earliest_ts = int(merged.loc[merged['anomaly'],'timestamp'].min())
            # select anomaly rows at that earliest timestamp (limit up to 5 later)
            rows = merged[(merged['anomaly']) & (merged['timestamp'] == earliest_ts)].copy()
            # compute breach side and severity_pct per spec (use value at that timestamp)
            def compute_breach_severity(r):
                val = r['value']
                p95 = r['P95']
                p5 = r['P5']
                breach = None
                severity = np.nan
                # high-side
                if pd.notna(p95) and val > p95:
                    breach = 'high-side'
                    if p95 != 0:
                        severity = 100.0 * (val - p95) / p95
                # low-side
                if pd.notna(p5) and val < p5:
                    # if both, take larger percent (compute low_pct and compare)
                    low_pct = np.nan
                    if p5 != 0:
                        low_pct = 100.0 * (p5 - val) / p5
                    if breach == 'high-side':
                        # compare high and low (unlikely both) and pick larger
                        high_pct = severity
                        if not np.isnan(low_pct) and low_pct > high_pct:
                            breach = 'low-side'
                            severity = low_pct
                    else:
                        breach = 'low-side'
                        severity = low_pct
                return pd.Series({'breach_side': breach, 'severity_pct': (float(severity) if pd.notna(severity) else np.nan)})

            bh = rows.apply(compute_breach_severity, axis=1)
            rows = pd.concat([rows, bh], axis=1)

            # Prepare final compact DataFrame with required columns, limit to at most 5 rows
            earliest_anomalies_df = rows[['timestamp','kpi_name','value','P95','P5','breach_side','severity_pct']].head(5).reset_index(drop=True)

# Display the result (compact)
earliest_anomalies_df

```
Out[8]:
```
Earliest anomaly(s) for emailservice-1 within the incident window (timestamp values as returned):

- Earliest anomaly timestamp: 1647768660
  1) kpi_name: container.node-6.container_cpu_cfs_periods
     - value: 87.5, P95: 81.5, P5: 55.0
     - breach: high-side
     - severity_pct: 7.36% (value is ~7.36% above P95)

  2) kpi_name: container.node-6.container_cpu_usage_seconds
     - value: 0.049974, P95: 0.047475, P5: 0.032083
     - breach: high-side
     - severity_pct: 5.26% (value is ~5.26% above P95)

Interpretation / suggested next steps (brief)
- Both earliest anomalies are CPU-related (cfs periods and CPU usage) on node-6 for emailservice-1 and are modest high-side breaches (≈5–7%).  
- Correlate these timestamps with the multi-minute mesh anomalies already found (emailservice→checkoutservice) and check emailservice-1 pod CPU scheduling/throttling, container limits, and pod logs around 1647768660.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp                                      kpi_name      value        P95         P5 breach_side  severity_pct
0  1647768660    container.node-6.container_cpu_cfs_periods  87.500000  81.500000  55.000000   high-side      7.361963
1  1647768660  container.node-6.container_cpu_usage_seconds   0.049974   0.047475   0.032083   high-side      5.263803```
```